[![pythonista](img/pythonista.png)](https://www.pythonista.io)

# Fechas, subconsultas y expresiones de tabla comunes.

## Configuración.

Ejecuta la siguiente celda antes que cualquier otra. Abre la base de datos del curso en modo de sólo lectura.

In [ ]:
import duckdb

%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.displaycon = False

conexion = duckdb.connect('data/thelook.duckdb', read_only=True)
%sql conexion

## Tipos de datos temporales.

| Tipo *ANSI* | Literal | Uso |
|---|---|---|
| ```DATE``` | ```DATE '2024-03-15'``` | Fecha de calendario |
| ```TIMESTAMP``` | ```TIMESTAMP '2024-03-15 16:30:00'``` | Fecha y hora |
| ```INTERVAL``` | ```INTERVAL '7' DAY``` | Duración |

En los datos del curso, columnas como ```created_at``` son de tipo ```TIMESTAMP```. Los literales se escriben con el nombre del tipo seguido de una cadena en formato *ISO 8601*.

In [ ]:
%%sql
SELECT DATE '2024-03-15' AS fecha,
  TIMESTAMP '2024-03-15 16:30:00' AS momento,
  DATE '2024-03-15' + INTERVAL '7' DAY AS una_semana_despues;

> **Nota de dialecto:** en *BigQuery*, ```TIMESTAMP``` representa un instante absoluto en *UTC* (equivale al ```TIMESTAMP WITH TIME ZONE``` del estándar), y su tipo ```DATETIME``` corresponde al ```TIMESTAMP``` estándar. En *SQLite* no existen tipos de fecha: se guardan como texto.

## Conversión y extracción.

* ```CAST(expresión AS tipo)``` convierte entre tipos; por ejemplo, de ```TIMESTAMP``` a ```DATE```.
* ```EXTRACT(parte FROM fecha)``` obtiene una parte de la fecha: ```YEAR```, ```MONTH```, ```DAY```, ```HOUR```, ```MINUTE```, ```SECOND```.
* ```CURRENT_DATE``` y ```CURRENT_TIMESTAMP``` regresan la fecha y el momento actuales.

In [ ]:
%%sql
SELECT order_id,
  created_at,
  CAST(created_at AS DATE) AS fecha,
  EXTRACT(YEAR FROM created_at) AS anio,
  EXTRACT(MONTH FROM created_at) AS mes,
  EXTRACT(HOUR FROM created_at) AS hora
FROM orders
ORDER BY order_id
FETCH FIRST 20 ROWS ONLY;

## Filtrar y agrupar por periodos.

* Órdenes por mes durante 2024. Para filtrar ```TIMESTAMP``` se recomienda un rango semiabierto (```>=``` inicio y ```<``` fin): a diferencia de ```BETWEEN '2024-01-01' AND '2024-12-31'```, no deja fuera las horas del último día.

In [ ]:
%%sql
SELECT EXTRACT(YEAR FROM created_at) AS anio,
  EXTRACT(MONTH FROM created_at) AS mes,
  COUNT(*) AS ordenes
FROM orders
WHERE created_at >= TIMESTAMP '2024-01-01 00:00:00'
  AND created_at < TIMESTAMP '2025-01-01 00:00:00'
GROUP BY EXTRACT(YEAR FROM created_at), EXTRACT(MONTH FROM created_at)
ORDER BY anio, mes;

> **Nota de dialecto:** casi todos los motores tienen una función para truncar una fecha al inicio de su periodo, pero su sintaxis varía: ```DATE_TRUNC('month', fecha)``` en *DuckDB*, *PostgreSQL* y *Snowflake*; ```DATE_TRUNC(fecha, MONTH)``` en *BigQuery*. Agrupar por ```EXTRACT``` funciona en todos.

## Aritmética de fechas.

Según el estándar:

* ```fecha + INTERVAL``` produce otra fecha.
* La resta de dos ```TIMESTAMP``` produce un ```INTERVAL```.

In [ ]:
%%sql
SELECT order_id,
  created_at,
  delivered_at,
  delivered_at - created_at AS tiempo_entrega,
  created_at + INTERVAL '30' DAY AS fin_garantia
FROM orders
WHERE delivered_at IS NOT NULL
ORDER BY order_id
FETCH FIRST 20 ROWS ONLY;

Para obtener la diferencia como un **número**, el estándar permite extraer los componentes de un ```INTERVAL``` con ```EXTRACT```:

* ```EXTRACT(DAY FROM intervalo)``` regresa los días completos.
* ```EXTRACT(HOUR FROM intervalo)``` regresa las horas restantes (de 0 a 23).

In [ ]:
%%sql
SELECT order_id,
  delivered_at - created_at AS tiempo_entrega,
  EXTRACT(DAY FROM (delivered_at - created_at)) AS dias,
  EXTRACT(HOUR FROM (delivered_at - created_at)) AS horas_restantes,
  EXTRACT(DAY FROM (delivered_at - created_at)) * 24
    + EXTRACT(HOUR FROM (delivered_at - created_at)) AS horas_totales
FROM orders
WHERE delivered_at IS NOT NULL
ORDER BY order_id
FETCH FIRST 10 ROWS ONLY;

* Tiempo promedio de entrega, en horas, para los 10 estados más lentos.

In [ ]:
%%sql
SELECT u.state,
  COUNT(*) AS ordenes_entregadas,
  ROUND(AVG(EXTRACT(DAY FROM (o.delivered_at - o.created_at)) * 24
            + EXTRACT(HOUR FROM (o.delivered_at - o.created_at))), 1) AS horas_promedio
FROM orders AS o
JOIN users AS u ON o.user_id = u.id
WHERE o.delivered_at IS NOT NULL
GROUP BY u.state
ORDER BY horas_promedio DESC
FETCH FIRST 10 ROWS ONLY;

> **Nota de dialecto:** la aritmética de fechas es una de las áreas donde los motores más difieren, y casi todos ofrecen funciones propias más cómodas: ```DATE_DIFF('hour', inicio, fin)``` en *DuckDB*; ```TIMESTAMP_DIFF(fin, inicio, HOUR)``` en *BigQuery*; ```DATEDIFF(hour, inicio, fin)``` en *SQL Server*; ```julianday(fin) - julianday(inicio)``` en *SQLite*. La forma con ```EXTRACT``` sobre un ```INTERVAL``` funciona en *DuckDB* y *PostgreSQL*.

## Subconsultas.

Una subconsulta es una consulta dentro de otra. Puede usarse como un valor, como una lista o como una tabla.

* Como valor: productos con precio mayor al promedio del catálogo.

In [ ]:
%%sql
SELECT name, retail_price
FROM products
WHERE retail_price > (
  SELECT AVG(retail_price)
  FROM products
)
ORDER BY retail_price
FETCH FIRST 20 ROWS ONLY;

* Como lista con ```IN```: clientes que han devuelto alguna orden.

In [ ]:
%%sql
SELECT id, first_name, last_name
FROM users
WHERE id IN (
  SELECT user_id
  FROM orders
  WHERE status = 'Returned'
)
ORDER BY id
FETCH FIRST 20 ROWS ONLY;

* Como tabla en ```FROM``` (tabla derivada): número promedio de órdenes por cliente.

In [ ]:
%%sql
SELECT ROUND(AVG(ordenes), 2) AS ordenes_promedio
FROM (
  SELECT user_id, COUNT(*) AS ordenes
  FROM orders
  GROUP BY user_id
) AS por_cliente;

## Expresiones de tabla comunes (```WITH```).

Las subconsultas anidadas se vuelven difíciles de leer. La cláusula ```WITH``` permite definir resultados intermedios con nombre (*CTE*, *Common Table Expressions*) y usarlos como si fueran tablas. **Es la forma estándar de escribir transformaciones en ingeniería de datos**: cada *CTE* es un paso legible del proceso.

* Paso 1: ventas por cliente. Paso 2: segmentar a los clientes. Paso 3: resumir por segmento.

In [ ]:
%%sql
WITH ventas_cliente AS (
  SELECT user_id,
    COUNT(DISTINCT order_id) AS ordenes,
    SUM(sale_price) AS total
  FROM order_items
  WHERE status = 'Complete'
  GROUP BY user_id
),
segmentos AS (
  SELECT user_id,
    total,
    CASE
      WHEN ordenes = 1 THEN 'Una compra'
      WHEN ordenes <= 3 THEN 'Ocasional'
      ELSE 'Frecuente'
    END AS segmento
  FROM ventas_cliente
)
SELECT segmento,
  COUNT(*) AS clientes,
  ROUND(AVG(total), 2) AS gasto_promedio
FROM segmentos
GROUP BY segmento
ORDER BY gasto_promedio DESC;

Observa que, al calcular el segmento en una *CTE*, la consulta final puede agrupar por ```segmento``` sin repetir la expresión ```CASE```.

## *CTE* contra subconsultas.

La misma pregunta se puede responder de ambas formas. **Categorías cuyas ventas superan el promedio de ventas por categoría:**

* Con subconsultas: se lee de adentro hacia afuera, y el cálculo de ventas por categoría aparece dos veces.

In [ ]:
%%sql
SELECT category, ventas
FROM (
  SELECT p.category, SUM(oi.sale_price) AS ventas
  FROM order_items AS oi
  JOIN products AS p ON oi.product_id = p.id
  GROUP BY p.category
) AS vc
WHERE ventas > (
  SELECT AVG(ventas)
  FROM (
    SELECT p.category, SUM(oi.sale_price) AS ventas
    FROM order_items AS oi
    JOIN products AS p ON oi.product_id = p.id
    GROUP BY p.category
  ) AS vc2
)
ORDER BY ventas DESC;

* Con una *CTE*: se lee de arriba hacia abajo, y el paso se define una sola vez y se usa dos veces.

In [ ]:
%%sql
WITH ventas_categoria AS (
  SELECT p.category, SUM(oi.sale_price) AS ventas
  FROM order_items AS oi
  JOIN products AS p ON oi.product_id = p.id
  GROUP BY p.category
)
SELECT category, ventas
FROM ventas_categoria
WHERE ventas > (SELECT AVG(ventas) FROM ventas_categoria)
ORDER BY ventas DESC;

| Aspecto | *CTE* (```WITH```) | Subconsulta |
|---|---|---|
| **Legibilidad** | ✅ Pasos con nombre, de arriba hacia abajo, como un *pipeline* | ❌ Anidada; se lee de adentro hacia afuera y se complica con cada nivel |
| **Reutilización** | ✅ Se define una vez y se referencia varias veces en la misma consulta | ❌ Hay que repetir el código, con riesgo de que las copias diverjan |
| **Depuración** | ✅ Se puede probar cada paso cambiando sólo el ```SELECT``` final | ❌ Hay que extraer la subconsulta para probarla |
| **Recursividad** | ✅ ```WITH RECURSIVE``` permite recorrer jerarquías (organigramas, categorías) | ❌ No es posible |
| **Alcance** | ⚠️ Sólo existe durante la consulta; no se guarda ni se comparte entre consultas (para eso están las vistas) | ⚠️ Igual |
| **Rendimiento** | ⚠️ En general equivalente: el optimizador suele integrar la *CTE* en la consulta. **En *BigQuery*, una *CTE* referenciada varias veces puede evaluarse una vez por referencia**, con su costo de tiempo, aunque los bytes facturados por tabla leída no se dupliquen. Algunos motores antiguos (*PostgreSQL* < 12) siempre la materializaban, lo que impedía optimizaciones | ⚠️ Equivalente |
| **Usos exclusivos** | | ✅ Valores escalares en ```SELECT``` o ```WHERE```, listas con ```IN``` y subconsultas **correlacionadas** con ```EXISTS``` |
| **Exceso** | ❌ Cadenas muy largas de *CTE* triviales pueden dispersar la lógica | ✅ Para un filtro sencillo y de un solo uso, una subconsulta corta es más directa |

**Regla práctica:** usa *CTE* para los pasos de una transformación y cuando un resultado se usa más de una vez. Usa subconsultas para filtros cortos (```IN```, ```EXISTS```, un valor escalar). Si un paso se necesita en varias consultas, conviértelo en una **vista** o una **tabla**.

### Subconsultas correlacionadas con ```EXISTS```.

Una subconsulta correlacionada hace referencia a la consulta exterior y se evalúa conceptualmente por cada fila. ```EXISTS``` regresa verdadero si la subconsulta encuentra al menos una fila.

* Clientes que han comprado al menos un producto de la categoría ```Jeans```.

In [ ]:
%%sql
SELECT u.id, u.first_name, u.last_name
FROM users AS u
WHERE EXISTS (
  SELECT 1
  FROM order_items AS oi
  JOIN products AS p ON oi.product_id = p.id
  WHERE oi.user_id = u.id
    AND p.category = 'Jeans'
)
ORDER BY u.id
FETCH FIRST 20 ROWS ONLY;

## ```UNION ALL``` y ```UNION DISTINCT```.

Los operadores de conjuntos apilan los resultados de dos consultas con las mismas columnas.

* ```UNION ALL``` conserva todas las filas (es más rápido).
* ```UNION DISTINCT``` elimina duplicados.
* ```INTERSECT DISTINCT``` y ```EXCEPT DISTINCT``` obtienen la intersección y la diferencia.

> **Nota de dialecto:** en el estándar, ```UNION``` sin calificativo equivale a ```UNION DISTINCT```. *BigQuery* exige escribir ```ALL``` o ```DISTINCT```, lo cual es una buena práctica en cualquier motor.

* Línea de tiempo de eventos de tres órdenes: creación, envío y entrega en una sola columna. Los eventos que no han ocurrido aparecen como ```NULL```.

In [ ]:
%%sql
WITH o AS (
  SELECT order_id, created_at, shipped_at, delivered_at
  FROM orders
  WHERE order_id IN (1, 2, 3)
)
SELECT order_id, 'creada' AS evento, created_at AS momento FROM o
UNION ALL
SELECT order_id, 'enviada', shipped_at FROM o
UNION ALL
SELECT order_id, 'entregada', delivered_at FROM o
ORDER BY order_id, momento;

## Ejercicios.

1. Calcula el número de clientes registrados por año (```users.created_at```).
2. Obtén las ventas (```sale_price```, estado ```Complete```) por mes durante 2024. ¿Cuál fue el mejor mes?
3. Calcula, en días, el tiempo promedio entre que una orden se crea y se envía (```shipped_at```), por año de creación.
4. Usando una subconsulta, obtén las categorías cuyo precio promedio es mayor que el precio promedio de todo el catálogo.
5. Usando ```WITH```, calcula las ventas mensuales de 2024 y después regresa sólo los meses cuyas ventas superan el promedio mensual de ese año.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2026.</p>